# Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. 
Tools are pairings of:
    
    1. A schema, including the name of the tool, a description, and/or argument definitions(often a JSON Schema).
    2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:llama-3.1-8b-instant")
response=model.invoke("Write me a essay on AI.")
response

AIMessage(content='**The Rise of Artificial Intelligence: A New Era of Innovation**\n\nArtificial Intelligence (AI) has been a topic of interest for decades, with its roots dating back to the mid-20th century. From its humble beginnings as a simple computer program that could play chess, to its current form as a sophisticated technology that can perform complex tasks, AI has evolved into a transformative force that is changing the world as we know it. In this essay, we will explore the history and development of AI, its current applications, and its potential impact on society.\n\n**The Early Years of AI**\n\nThe concept of AI was first introduced in 1956 by computer scientist John McCarthy, who organized a conference at Dartmouth College that brought together experts from various fields to discuss the possibility of creating machines that could think and learn like humans. The term "Artificial Intelligence" was coined during this conference, and it marked the beginning of a new era in

In [2]:
## Tools

from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location """
    return f"It's sunny in {location}"

model_with_tools=model.bind_tools([get_weather])

In [4]:
response=model_with_tools.invoke("What's the weather in Bangalore?")
print(response)

content='' additional_kwargs={'tool_calls': [{'id': 'ybm4z8mgg', 'function': {'arguments': '{"location":"Bangalore"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 219, 'total_tokens': 234, 'completion_time': 0.02732756, 'completion_tokens_details': None, 'prompt_time': 0.013304732, 'prompt_tokens_details': None, 'queue_time': 0.089116174, 'total_time': 0.040632292}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d2066-57e9-7171-8b02-e653c3420f91-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bangalore'}, 'id': 'ybm4z8mgg', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 219, 'output_tokens': 15, 'total_tokens': 234}


In [5]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Bangalore'},
  'id': 'ybm4z8mgg',
  'type': 'tool_call'}]

## Tool Execution Loop

In [9]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Bangalore?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

In [10]:
messages

[{'role': 'user', 'content': "What's the weather in Bangalore?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'at1facah5', 'function': {'arguments': '{"location":"Bangalore"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 219, 'total_tokens': 234, 'completion_time': 0.029070107, 'completion_tokens_details': None, 'prompt_time': 0.01340729, 'prompt_tokens_details': None, 'queue_time': 0.018402209, 'total_time': 0.042477397}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_6a1eabf260', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d2074-fd0b-7f21-b01e-4059f3781e45-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bangalore'}, 'id': 'at1facah5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 219, 'output_tokens': 15, 'total_tokens': 234}),
 ToolMessage(content=

In [11]:
model_with_tools

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001C9621E82C0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C963318EF0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather at a location', 'parameters': {'properties': {'location': {'type': 'string'}}, 'required': ['location'], 'type': 'object'}}}]}, config={}, config_factories=[])